# Final Load Prep For M4

This notebook prepares a customer-level KPI dataset and lightweight segment fields for downstream Tableau and statistical analysis. It reads only the validated M2 output, engineers analysis-friendly columns, validates readiness, and saves the result to `data/processed/final_kpi_data.csv`.


In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')


## 1. Load Cleaned Data

In [2]:
data_path = '../data/processed/cleaned_data.csv'
df = pd.read_csv(data_path)
print('Shape:', df.shape)
df.head()


Shape: (3150, 13)


,id,subscription_length,charge_amount,seconds_of_use,frequency_of_use,frequency_of_sms,distinct_called_numbers,call_failures,tariff_plan,status,age_group,complaints,churn
0,1,35,0,1420,41,36,20,1,A,Active,30-40,N,0
1,2,28,0,920,32,20,12,7,A,Active,Under 30,N,0
2,3,40,0,88,6,8,6,0,A,Inactive,30-40,N,1
3,4,38,0,13963,170,9,47,9,A,Active,30-40,N,0
4,5,38,0,13773,169,0,44,7,A,Active,30-40,N,0


## 2. Engineer KPI And Segment Fields

In [3]:
df_kpi = df.copy()

df_kpi['usage_intensity'] = df_kpi['seconds_of_use'] / df_kpi['subscription_length']
df_kpi['engagement_intensity'] = df_kpi['frequency_of_use'] / df_kpi['subscription_length']
df_kpi['sms_intensity'] = df_kpi['frequency_of_sms'] / df_kpi['subscription_length']
df_kpi['call_failure_rate'] = np.where(
    df_kpi['frequency_of_use'] > 0,
    df_kpi['call_failures'] / df_kpi['frequency_of_use'],
    0
)
df_kpi['network_reach_ratio'] = np.where(
    df_kpi['frequency_of_use'] > 0,
    df_kpi['distinct_called_numbers'] / df_kpi['frequency_of_use'],
    0
)

usage_cut = df_kpi['usage_intensity'].median()
tenure_cut = df_kpi['subscription_length'].median()
failure_cut = df_kpi['call_failure_rate'].median()

df_kpi['usage_segment'] = np.where(df_kpi['usage_intensity'] >= usage_cut, 'High usage', 'Low usage')
df_kpi['charge_segment'] = np.where(df_kpi['charge_amount'] > 0, 'Positive charge tier', 'Zero charge tier')
df_kpi['tenure_segment'] = np.where(df_kpi['subscription_length'] >= tenure_cut, 'Long tenure', 'Short tenure')
df_kpi['failure_segment'] = np.where(df_kpi['call_failure_rate'] >= failure_cut, 'High failure', 'Low failure')

df_kpi['tariff_plan_flag'] = np.where(df_kpi['tariff_plan'] == 'B', 1, 0)
df_kpi['status_inactive_flag'] = np.where(df_kpi['status'] == 'Inactive', 1, 0)
df_kpi['complaints_flag'] = np.where(df_kpi['complaints'] == 'Y', 1, 0)
df_kpi['age_group_code'] = df_kpi['age_group'].map({'Under 30': 0, '30-40': 1, 'Over 40': 2})

df_kpi.head()


,id,subscription_length,charge_amount,seconds_of_use,frequency_of_use,frequency_of_sms,distinct_called_numbers,call_failures,tariff_plan,status,age_group,complaints,churn,usage_intensity,engagement_intensity,sms_intensity,call_failure_rate,network_reach_ratio,usage_segment,charge_segment,tenure_segment,failure_segment,tariff_plan_flag,status_inactive_flag,complaints_flag,age_group_code
0,1,35,0,1420,41,36,20,1,A,Active,30-40,N,0,40.571,1.171,1.029,0.024,0.488,Low usage,Zero charge tier,Long tenure,Low failure,0,0,0,1
1,2,28,0,920,32,20,12,7,A,Active,Under 30,N,0,32.857,1.143,0.714,0.219,0.375,Low usage,Zero charge tier,Short tenure,High failure,0,0,0,0
2,3,40,0,88,6,8,6,0,A,Inactive,30-40,N,1,2.200,0.150,0.200,0.000,1.000,Low usage,Zero charge tier,Long tenure,Low failure,0,1,0,1
3,4,38,0,13963,170,9,47,9,A,Active,30-40,N,0,367.447,4.474,0.237,0.053,0.276,High usage,Zero charge tier,Long tenure,Low failure,0,0,0,1
4,5,38,0,13773,169,0,44,7,A,Active,30-40,N,0,362.447,4.447,0.000,0.041,0.260,High usage,Zero charge tier,Long tenure,Low failure,0,0,0,1


## 2A. Dataset Handoff Strategy

This phase now produces **two M3 outputs** for two different downstream uses:

- **`final_kpi_data.csv`** keeps business-facing fields for Tableau, KPI tracking, and operational monitoring.
- **`final_model_data.csv`** removes leakage-prone fields before M4 modeling so that statistical analysis is less likely to overstate predictive power.

The following columns are excluded from the modeling dataset because they may reflect events that occur very close to, or after, the churn outcome:

- `status`
- `status_inactive_flag`
- `complaints`
- `complaints_flag`

These fields are still useful for dashboards and retention operations, but they should not be treated as safely predictive without a time-window review.


In [4]:
leakage_columns = ['status', 'status_inactive_flag', 'complaints', 'complaints_flag']
df_model = df_kpi.drop(columns=leakage_columns).copy()

print('KPI dataset shape:', df_kpi.shape)
print('Modeling dataset shape:', df_model.shape)
print('Removed leakage-prone columns:', leakage_columns)

df_model.head()


KPI dataset shape: (3150, 26)
Modeling dataset shape: (3150, 22)
Removed leakage-prone columns: ['status', 'status_inactive_flag', 'complaints', 'complaints_flag']


,id,subscription_length,charge_amount,seconds_of_use,frequency_of_use,frequency_of_sms,distinct_called_numbers,call_failures,tariff_plan,age_group,churn,usage_intensity,engagement_intensity,sms_intensity,call_failure_rate,network_reach_ratio,usage_segment,charge_segment,tenure_segment,failure_segment,tariff_plan_flag,age_group_code
0,1,35,0,1420,41,36,20,1,A,30-40,0,40.571,1.171,1.029,0.024,0.488,Low usage,Zero charge tier,Long tenure,Low failure,0,1
1,2,28,0,920,32,20,12,7,A,Under 30,0,32.857,1.143,0.714,0.219,0.375,Low usage,Zero charge tier,Short tenure,High failure,0,0
2,3,40,0,88,6,8,6,0,A,30-40,1,2.200,0.150,0.200,0.000,1.000,Low usage,Zero charge tier,Long tenure,Low failure,0,1
3,4,38,0,13963,170,9,47,9,A,30-40,0,367.447,4.474,0.237,0.053,0.276,High usage,Zero charge tier,Long tenure,Low failure,0,1
4,5,38,0,13773,169,0,44,7,A,30-40,0,362.447,4.447,0.000,0.041,0.260,High usage,Zero charge tier,Long tenure,Low failure,0,1


## 3. Create M4 Support Tables

In [5]:
segment_summary = (df_kpi.groupby(['usage_segment', 'failure_segment', 'tenure_segment'])['churn']
                   .agg(['count', 'mean'])
                   .rename(columns={'count': 'customers', 'mean': 'churn_rate'})
                   .reset_index()
                   .sort_values('churn_rate', ascending=False))

category_summary = {
    col: (df_kpi.groupby(col)['churn']
          .agg(['count', 'mean'])
          .rename(columns={'count': 'customers', 'mean': 'churn_rate'})
          .reset_index()
          .sort_values('churn_rate', ascending=False))
    for col in ['tariff_plan', 'status', 'age_group', 'complaints', 'usage_segment', 'charge_segment', 'tenure_segment', 'failure_segment']
}

segment_summary.head(10)


,usage_segment,failure_segment,tenure_segment,customers,churn_rate
4,Low usage,High failure,Long tenure,568,0.301
5,Low usage,High failure,Short tenure,333,0.240
7,Low usage,Low failure,Short tenure,364,0.209
6,Low usage,Low failure,Long tenure,310,0.194
1,High usage,High failure,Short tenure,282,0.177
3,High usage,Low failure,Short tenure,466,0.077
0,High usage,High failure,Long tenure,392,0.048
2,High usage,Low failure,Long tenure,435,0.007


## 4. Validate M4 Readiness

In [6]:
null_check = df_kpi.isna().sum().sort_values(ascending=False)
null_check.head(10)


id                      0
subscription_length     0
complaints_flag         0
status_inactive_flag    0
tariff_plan_flag        0
failure_segment         0
tenure_segment          0
charge_segment          0
usage_segment           0
network_reach_ratio     0

In [7]:
print('KPI dataset null counts:')
display(df_kpi.isnull().sum())
print('KPI dataset duplicate rows:', df_kpi.duplicated().sum())

print('\nModeling dataset null counts:')
display(df_model.isnull().sum())
print('Modeling dataset duplicate rows:', df_model.duplicated().sum())


KPI dataset null counts:
KPI dataset duplicate rows: 0

Modeling dataset null counts:
Modeling dataset duplicate rows: 0


id                         0
subscription_length        0
charge_amount              0
seconds_of_use             0
frequency_of_use           0
frequency_of_sms           0
distinct_called_numbers    0
call_failures              0
tariff_plan                0
status                     0
age_group                  0
complaints                 0
churn                      0
usage_intensity            0
engagement_intensity       0
sms_intensity              0
call_failure_rate          0
network_reach_ratio        0
usage_segment              0
charge_segment             0
tenure_segment             0
failure_segment            0
tariff_plan_flag           0
status_inactive_flag       0
complaints_flag            0
age_group_code             0

id                         0
subscription_length        0
charge_amount              0
seconds_of_use             0
frequency_of_use           0
frequency_of_sms           0
distinct_called_numbers    0
call_failures              0
tariff_plan                0
age_group                  0
churn                      0
usage_intensity            0
engagement_intensity       0
sms_intensity              0
call_failure_rate          0
network_reach_ratio        0
usage_segment              0
charge_segment             0
tenure_segment             0
failure_segment            0
tariff_plan_flag           0
age_group_code             0

In [8]:
numeric_ready_cols = [
    'subscription_length', 'charge_amount', 'seconds_of_use', 'frequency_of_use',
    'frequency_of_sms', 'distinct_called_numbers', 'call_failures', 'churn',
    'usage_intensity', 'engagement_intensity', 'sms_intensity', 'call_failure_rate',
    'network_reach_ratio', 'tariff_plan_flag', 'status_inactive_flag', 'complaints_flag', 'age_group_code'
]

print('Any null values:', df_kpi.isna().any().any())
print('Numeric columns ready:', df_kpi[numeric_ready_cols].select_dtypes(include=[np.number]).shape[1] == len(numeric_ready_cols))
print('Categorical columns:', df_kpi.select_dtypes(include='object').columns.tolist())

df_kpi[numeric_ready_cols].describe().T


Any null values: False
Numeric columns ready: True
Categorical columns: ['tariff_plan', 'status', 'age_group', 'complaints', 'usage_segment', 'charge_segment', 'tenure_segment', 'failure_segment']


,count,mean,std,min,25%,50%,75%,max
subscription_length,"3,150.000",32.542,8.573,3.000,30.000,35.000,38.000,47.000
charge_amount,"3,150.000",0.943,1.521,0.000,0.000,0.000,1.000,10.000
seconds_of_use,"3,150.000","4,472.460","4,197.909",0.000,"1,391.250","2,990.000","6,478.250","17,090.000"
frequency_of_use,"3,150.000",69.461,57.413,0.000,27.000,54.000,95.000,255.000
frequency_of_sms,"3,150.000",73.175,112.238,0.000,6.000,21.000,87.000,522.000
distinct_called_numbers,"3,150.000",23.510,17.217,0.000,10.000,21.000,34.000,97.000
call_failures,"3,150.000",7.628,7.264,0.000,1.000,6.000,12.000,36.000
churn,"3,150.000",0.157,0.364,0.000,0.000,0.000,0.000,1.000
usage_intensity,"3,150.000",152.664,159.054,0.000,48.557,97.041,201.865,"1,376.667"
engagement_intensity,"3,150.000",2.359,2.346,0.000,0.973,1.716,2.866,22.143


## 5. Save Final KPI Dataset

In [9]:
output_kpi_path = '../data/processed/final_kpi_data.csv'
output_model_path = '../data/processed/final_model_data.csv'

df_kpi.to_csv(output_kpi_path, index=False)
df_model.to_csv(output_model_path, index=False)

print(f'Saved KPI dataset: {output_kpi_path}')
print(f'Saved modeling dataset: {output_model_path}')
print('Final KPI shape:', df_kpi.shape)
print('Final modeling shape:', df_model.shape)


Saved KPI dataset: ../data/processed/final_kpi_data.csv
Saved modeling dataset: ../data/processed/final_model_data.csv
Final KPI shape: (3150, 26)
Final modeling shape: (3150, 22)


**Readiness note**

`df_kpi` is customer-level, null-checked, and structured for both Tableau and M4 statistical work. It preserves the original cleaned fields, adds telecom-relevant KPI ratios, and includes simple encoded columns for modeling-ready numeric inputs.
